In [1]:
import pandas as pd
import chardet
import os

In [2]:
data_customers_df = pd.read_csv('olist_customers_dataset.csv')
data_geolocation_df = pd.read_csv('olist_geolocation_dataset.csv')
data_order_items_df = pd.read_csv('olist_order_items_dataset.csv')
data_order_payments_df = pd.read_csv('olist_order_payments_dataset.csv')
data_order_reviews_df = pd.read_csv('olist_order_reviews_dataset.csv')
data_orders_df = pd.read_csv('olist_orders_dataset.csv')
data_products_df = pd.read_csv('olist_products_dataset.csv')
data_sellers_df = pd.read_csv('olist_sellers_dataset.csv')
data_category_name_translation_df = pd.read_csv('product_category_name_translation.csv')

In [ ]:
class DataCleaning:
    def __init__(self, data):
        # Si 'data' es un string, lo tratamos como ruta de archivo
        if isinstance(data, str):
            self.ruta = data
            self.nombre = os.path.basename(data)
            self.df = None
        # Si 'data' es un DataFrame, lo asignamos directamente
        elif isinstance(data, pd.DataFrame):
            self.ruta = None
            self.nombre = "DataFrame_en_Memoria"
            self.df = data.copy() # Usamos .copy() para no modificar el original
        self.encoding = None

    def visualizar_muestra(self, n=5):
        """Muestra las primeras n filas del dataset"""
        if self.df is not None:
            print(f"\n👀 Muestra de {self.nombre}:")
            display(self.df.head(n))
        else:
            print("⚠️ No hay datos cargados para mostrar la muestra.")
        return self
    
    # === PUNTO 1: INSPECCIÓN INICIAL Y CARGA ===
    def validar_y_cargar(self):
        """Si ya tenemos el df cargado, solo retornamos self"""
        if self.df is not None:
            print(f"✅ [PUNTO 1] Datos ya cargados en memoria ({self.nombre}).")
            return self
        
        # Si no, procedemos con la lógica original de carga de archivos
        if not os.path.exists(self.ruta):
            print(f"❌ Error: El archivo {self.nombre} no existe.")
            return self
            
        with open(self.ruta, 'rb') as f:
            resultado = chardet.detect(f.read(25000))
        
        self.encoding = resultado['encoding']
        self.df = pd.read_csv(self.ruta, encoding=self.encoding)
        print(f"✅ [PUNTO 1] {self.nombre} cargado con éxito. Encoding: {self.encoding.upper()}")
        return self
    
    def revisar_dimensiones(self):
        """Muestra el tamaño actual del dataset"""
        if self.df is not None:
            filas, columnas = self.df.shape
            print(f"📊 [PUNTO 1] Dimensiones: {filas} filas y {columnas} columnas.")
        else:
            print("⚠️ No hay datos cargados aún.")
        return self
    
    def revisar_tipos_datos(self):
        """
        [PUNTO 1] Muestra un resumen de los tipos de datos actuales 
        y detecta posibles columnas candidatas a corregir.
        """
        if self.df is None:
            print("⚠️ No hay datos cargados para revisar los tipos.")
            return self
            
        print(f"\n📋 [PUNTO 1] Diagnóstico de Tipos de Datos para: {self.nombre}")
        print("-" * 60)
        
        # Recorremos cada columna para dar un reporte detallado
        for col in self.df.columns:
            tipo = self.df[col].dtype
            nulos = self.df[col].isnull().sum()
            porcentaje_nulos = (nulos / len(self.df)) * 100
            
            # Formateo visual limpio
            print(f"🔹 Columna: {col:<30} | Tipo: {str(tipo):<10} | Nulos: {nulos} ({porcentaje_nulos:.1f}%)")
            
        print("-" * 60)
        return self
    
    def corregir_tipo_datos(self):
        """
        Detecta y convierte tipos automáticamente: 
        1. Fechas (analizando el formato)
        2. Numéricos (limpiando caracteres extraños)
        """
        for col in self.df.columns:
            if self.df[col].dtype == 'object':
                # 1. Intentar fecha
                date_attempt = pd.to_datetime(self.df[col], errors='coerce')
                if date_attempt.notnull().mean() > 0.7:
                    self.df[col] = date_attempt
                    continue
                
                # 2. Intentar numérico (Flotantes y Enteros)
                # Primero limpiamos: quitamos comas, espacios, y signos de moneda si existen
                cleaned_col = self.df[col].astype(str).str.replace(',', '').str.replace(' ', '')
                
                # Intentamos convertir
                num_attempt = pd.to_numeric(cleaned_col, errors='coerce')
                
                # Si el porcentaje de éxito es alto (> 70%)
                if num_attempt.notnull().mean() > 0.7:
                    self.df[col] = num_attempt

                # 3. Si llega aquí, es String puro -> Limpiar y Optimizar
                self.df[col] = self.df[col].astype(str).str.strip().fillna('N/A')
                
                # 4. Optimización de cardinalidad (solo si no es demasiado único)
                num_unique = self.df[col].nunique()
                if num_unique / len(self.df) < 0.5:
                    self.df[col] = self.df[col].astype('category')
        return self
    
    def estandarizar_columnas(self):
        """
        Limpia y estandariza los nombres de las columnas:
        1. Quita espacios al inicio/final.
        2. Convierte a minúsculas.
        3. Reemplaza espacios internos por guiones bajos (snake_case).
        """
        # 1. strip() para espacios invisibles
        # 2. str.lower() para minúsculas
        # 3. str.replace(' ', '_') para formato snake_case
        self.df.columns = (
            self.df.columns.str.strip()
            .str.lower()
            .str.replace(' ', '_')
            .str.replace('-', '_') # También reemplazamos guiones por si acaso
        )
        print("Columnas estandarizadas a formato snake_case.")
        return self      
    

In [40]:
ordenes_cleaner = DataCleaning('olist_orders_dataset.csv')


df_orders_perfecto = (ordenes_cleaner
                      .validar_y_cargar()
                      .revisar_dimensiones()
                      .revisar_tipos_datos()
                      .corregir_tipo_datos()
                      .estandarizar_columnas()
                      .revisar_tipos_datos()
                      .visualizar_muestra()
                      )

✅ [PUNTO 1] olist_orders_dataset.csv cargado con éxito. Encoding: ASCII
📊 [PUNTO 1] Dimensiones: 99441 filas y 8 columnas.

📋 [PUNTO 1] Diagnóstico de Tipos de Datos para: olist_orders_dataset.csv
------------------------------------------------------------
🔹 Columna: order_id                       | Tipo: object     | Nulos: 0 (0.0%)
🔹 Columna: customer_id                    | Tipo: object     | Nulos: 0 (0.0%)
🔹 Columna: order_status                   | Tipo: object     | Nulos: 0 (0.0%)
🔹 Columna: order_purchase_timestamp       | Tipo: object     | Nulos: 0 (0.0%)
🔹 Columna: order_approved_at              | Tipo: object     | Nulos: 160 (0.2%)
🔹 Columna: order_delivered_carrier_date   | Tipo: object     | Nulos: 1783 (1.8%)
🔹 Columna: order_delivered_customer_date  | Tipo: object     | Nulos: 2965 (3.0%)
🔹 Columna: order_estimated_delivery_date  | Tipo: object     | Nulos: 0 (0.0%)
------------------------------------------------------------


C:\Users\Sebastian\AppData\Local\Temp\ipykernel_16200\2570439853.py:86: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date_attempt = pd.to_datetime(self.df[col], errors='coerce');
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_16200\2570439853.py:86: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date_attempt = pd.to_datetime(self.df[col], errors='coerce');


Columnas estandarizadas a formato snake_case.

📋 [PUNTO 1] Diagnóstico de Tipos de Datos para: olist_orders_dataset.csv
------------------------------------------------------------
🔹 Columna: order_id                       | Tipo: object     | Nulos: 0 (0.0%)
🔹 Columna: customer_id                    | Tipo: object     | Nulos: 0 (0.0%)
🔹 Columna: order_status                   | Tipo: category   | Nulos: 0 (0.0%)
🔹 Columna: order_purchase_timestamp       | Tipo: datetime64[ns] | Nulos: 0 (0.0%)
🔹 Columna: order_approved_at              | Tipo: datetime64[ns] | Nulos: 160 (0.2%)
🔹 Columna: order_delivered_carrier_date   | Tipo: datetime64[ns] | Nulos: 1783 (1.8%)
🔹 Columna: order_delivered_customer_date  | Tipo: datetime64[ns] | Nulos: 2965 (3.0%)
🔹 Columna: order_estimated_delivery_date  | Tipo: datetime64[ns] | Nulos: 0 (0.0%)
------------------------------------------------------------

👀 Muestra de olist_orders_dataset.csv:


C:\Users\Sebastian\AppData\Local\Temp\ipykernel_16200\2570439853.py:86: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date_attempt = pd.to_datetime(self.df[col], errors='coerce');


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [44]:
ordenes_cleaner_df = DataCleaning(data_orders_df)
customer_cleaner_df = DataCleaning(data_customers_df)
geolocation_cleaner_df = DataCleaning(data_geolocation_df)
order_items_cleaner_df = DataCleaning(data_order_items_df)
order_payments_cleaner_df = DataCleaning(data_order_payments_df)
order_reviews_cleaner_df = DataCleaning(data_order_reviews_df)
products_cleaner_df = DataCleaning(data_products_df)
sellers_cleaner_df = DataCleaning(data_sellers_df)
category_name_translation_cleaner_df = DataCleaning(data_category_name_translation_df)


df_orders_perfecto = (ordenes_cleaner_df
                      .validar_y_cargar()
                      .revisar_dimensiones()
                      .revisar_tipos_datos()
                      .corregir_tipo_datos()
                      .estandarizar_columnas()
                      .revisar_tipos_datos()
                      .visualizar_muestra()
                      )

df_customers_perfecto = (customer_cleaner_df
                         .validar_y_cargar()
                         .revisar_dimensiones()
                         .revisar_tipos_datos()
                         .corregir_tipo_datos()
                         .estandarizar_columnas()
                         .revisar_tipos_datos()
                         .visualizar_muestra()
                         )

df_geolocation_perfecto = (geolocation_cleaner_df
                            .validar_y_cargar()
                            .revisar_dimensiones()
                            .revisar_tipos_datos()
                            .corregir_tipo_datos()
                            .estandarizar_columnas()
                            .revisar_tipos_datos()
                            .visualizar_muestra()
                            )

df_order_items_perfecto = (order_items_cleaner_df
                            .validar_y_cargar()
                            .revisar_dimensiones()
                            .revisar_tipos_datos()
                            .corregir_tipo_datos()
                            .estandarizar_columnas()
                            .revisar_tipos_datos()
                            .visualizar_muestra()
                            )

df_order_payments_perfecto = (order_payments_cleaner_df
                            .validar_y_cargar()
                            .revisar_dimensiones()
                            .revisar_tipos_datos()
                            .corregir_tipo_datos()
                            .estandarizar_columnas()
                            .revisar_tipos_datos()
                            .visualizar_muestra()
                            )

df_order_reviews_perfecto = (order_reviews_cleaner_df
                            .validar_y_cargar()
                            .revisar_dimensiones()
                            .revisar_tipos_datos()
                            .corregir_tipo_datos()
                            .estandarizar_columnas()
                            .revisar_tipos_datos()
                            .visualizar_muestra()
                            )

df_products_perfecto = (products_cleaner_df
                            .validar_y_cargar()
                            .revisar_dimensiones()
                            .revisar_tipos_datos()
                            .corregir_tipo_datos()
                            .estandarizar_columnas()
                            .revisar_tipos_datos()
                            .visualizar_muestra()
                            )

df_sellers_perfecto = (sellers_cleaner_df
                            .validar_y_cargar()
                            .revisar_dimensiones()
                            .revisar_tipos_datos()
                            .corregir_tipo_datos()
                            .estandarizar_columnas()
                            .revisar_tipos_datos()
                            .visualizar_muestra()
                            )

df_category_name_translation_perfecto = (category_name_translation_cleaner_df
                            .validar_y_cargar()
                            .revisar_dimensiones()
                            .revisar_tipos_datos()
                            .corregir_tipo_datos()
                            .estandarizar_columnas()
                            .revisar_tipos_datos()
                            .visualizar_muestra()
                            )

✅ [PUNTO 1] Datos ya cargados en memoria (DataFrame_en_Memoria).
📊 [PUNTO 1] Dimensiones: 99441 filas y 8 columnas.

📋 [PUNTO 1] Diagnóstico de Tipos de Datos para: DataFrame_en_Memoria
------------------------------------------------------------
🔹 Columna: order_id                       | Tipo: object     | Nulos: 0 (0.0%)
🔹 Columna: customer_id                    | Tipo: object     | Nulos: 0 (0.0%)
🔹 Columna: order_status                   | Tipo: object     | Nulos: 0 (0.0%)
🔹 Columna: order_purchase_timestamp       | Tipo: object     | Nulos: 0 (0.0%)
🔹 Columna: order_approved_at              | Tipo: object     | Nulos: 160 (0.2%)
🔹 Columna: order_delivered_carrier_date   | Tipo: object     | Nulos: 1783 (1.8%)
🔹 Columna: order_delivered_customer_date  | Tipo: object     | Nulos: 2965 (3.0%)
🔹 Columna: order_estimated_delivery_date  | Tipo: object     | Nulos: 0 (0.0%)
------------------------------------------------------------


C:\Users\Sebastian\AppData\Local\Temp\ipykernel_16200\2570439853.py:86: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date_attempt = pd.to_datetime(self.df[col], errors='coerce');
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_16200\2570439853.py:86: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date_attempt = pd.to_datetime(self.df[col], errors='coerce');


Columnas estandarizadas a formato snake_case.

📋 [PUNTO 1] Diagnóstico de Tipos de Datos para: DataFrame_en_Memoria
------------------------------------------------------------
🔹 Columna: order_id                       | Tipo: object     | Nulos: 0 (0.0%)
🔹 Columna: customer_id                    | Tipo: object     | Nulos: 0 (0.0%)
🔹 Columna: order_status                   | Tipo: category   | Nulos: 0 (0.0%)
🔹 Columna: order_purchase_timestamp       | Tipo: datetime64[ns] | Nulos: 0 (0.0%)
🔹 Columna: order_approved_at              | Tipo: datetime64[ns] | Nulos: 160 (0.2%)
🔹 Columna: order_delivered_carrier_date   | Tipo: datetime64[ns] | Nulos: 1783 (1.8%)
🔹 Columna: order_delivered_customer_date  | Tipo: datetime64[ns] | Nulos: 2965 (3.0%)
🔹 Columna: order_estimated_delivery_date  | Tipo: datetime64[ns] | Nulos: 0 (0.0%)
------------------------------------------------------------

👀 Muestra de DataFrame_en_Memoria:


C:\Users\Sebastian\AppData\Local\Temp\ipykernel_16200\2570439853.py:86: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date_attempt = pd.to_datetime(self.df[col], errors='coerce');


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


...

C:\Users\Sebastian\AppData\Local\Temp\ipykernel_16200\2570439853.py:86: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date_attempt = pd.to_datetime(self.df[col], errors='coerce');
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_16200\2570439853.py:86: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date_attempt = pd.to_datetime(self.df[col], errors='coerce');
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_16200\2570439853.py:86: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date_attempt = pd.to_datetime(self.df[col], errors='coerce');


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


✅ [PUNTO 1] Datos ya cargados en memoria (DataFrame_en_Memoria).
📊 [PUNTO 1] Dimensiones: 71 filas y 2 columnas.

📋 [PUNTO 1] Diagnóstico de Tipos de Datos para: DataFrame_en_Memoria
------------------------------------------------------------
🔹 Columna: product_category_name          | Tipo: object     | Nulos: 0 (0.0%)
🔹 Columna: product_category_name_english  | Tipo: object     | Nulos: 0 (0.0%)
------------------------------------------------------------
Columnas estandarizadas a formato snake_case.

📋 [PUNTO 1] Diagnóstico de Tipos de Datos para: DataFrame_en_Memoria
------------------------------------------------------------
🔹 Columna: product_category_name          | Tipo: object     | Nulos: 0 (0.0%)
🔹 Columna: product_category_name_english  | Tipo: object     | Nulos: 0 (0.0%)
------------------------------------------------------------

👀 Muestra de DataFrame_en_Memoria:


C:\Users\Sebastian\AppData\Local\Temp\ipykernel_16200\2570439853.py:86: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date_attempt = pd.to_datetime(self.df[col], errors='coerce');
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_16200\2570439853.py:86: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date_attempt = pd.to_datetime(self.df[col], errors='coerce');


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [46]:
df_final_orders_df = df_orders_perfecto.df
df_final_orders_df.info()

df_final_customers_df = df_customers_perfecto.df
df_final_customers_df.info()

df_final_geolocation_df = df_geolocation_perfecto.df
df_final_geolocation_df.info()

df_final_order_items_df = df_order_items_perfecto.df
df_final_order_items_df.info()

df_final_order_payments_df = df_order_payments_perfecto.df
df_final_order_payments_df.info()

df_final_order_reviews_df = df_order_reviews_perfecto.df
df_final_order_reviews_df.info()

df_final_products_df = df_products_perfecto.df
df_final_products_df.info()

df_final_sellers_df = df_sellers_perfecto.df
df_final_sellers_df.info()

df_final_category_name_translation_df = df_category_name_translation_perfecto.df
df_final_category_name_translation_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  category      
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: category(1), datetime64[ns](5), object(2)
memory usage: 5.4+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Co

In [47]:
output_folder = 'data_limpia'
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"📁 Carpeta '{output_folder}' creada.")

# 2. Crear un diccionario con tus objetos ya procesados
# Esto mapea el nombre que quieres para el archivo con el objeto de tu clase
datasets = {
    'orders': df_orders_perfecto,
    'customers': df_customers_perfecto,
    'geolocation': df_geolocation_perfecto,
    'order_items': df_order_items_perfecto,
    'order_payments': df_order_payments_perfecto,
    'order_reviews': df_order_reviews_perfecto,
    'products': df_products_perfecto,
    'sellers': df_sellers_perfecto,
    'category_translation': df_category_name_translation_perfecto
}

# 3. Iterar y guardar
for name, obj in datasets.items():
    file_path = os.path.join(output_folder, f'{name}_limpio.csv')
    obj.df.to_csv(file_path, index=False)
    print(f"✅ Guardado: {file_path}")

📁 Carpeta 'data_limpia' creada.
✅ Guardado: data_limpia\orders_limpio.csv
✅ Guardado: data_limpia\customers_limpio.csv
✅ Guardado: data_limpia\geolocation_limpio.csv
✅ Guardado: data_limpia\order_items_limpio.csv
✅ Guardado: data_limpia\order_payments_limpio.csv
✅ Guardado: data_limpia\order_reviews_limpio.csv
✅ Guardado: data_limpia\products_limpio.csv
✅ Guardado: data_limpia\sellers_limpio.csv
✅ Guardado: data_limpia\category_translation_limpio.csv
